In [ ]:
sample = 'mouse_skin'
save_dir = 'size_intensity_sen_vs_nsen'

In [ ]:
import cv2
import glob
import numpy as np
import pandas as pd
import scipy.io
from scipy.ndimage import rotate
from skimage import transform as tf
from skimage.transform import warp
import scipy.io
import cv2
import matplotlib.pyplot as plt

all_cell_mapping = pd.read_csv(f'./{sample}/all_cell_mapping.csv', index_col=0)
all_cell_raman = pd.read_csv(f'./{sample}/all_cell_raman.csv', index_col=0)

In [ ]:
# Old sample, S vs NS, size

from scipy import stats
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

label_sets = ['p21+']
tiltle_sets = ['p21']

for label_set, tiltle_set in zip(label_sets, tiltle_sets):
   
    selected_cell_mapping = all_cell_mapping[all_cell_mapping['sample_type'] == 'O']

    if len(selected_cell_mapping) < 3:
        continue

    labels = selected_cell_mapping[label_set].values * 1 # S vs NS
    features = selected_cell_mapping['num_pixel']

    positive = features[labels == 1]
    negative = features[labels == 0]
    
    if len(positive) < 3 or len(negative) < 3:
        continue

    # Perform Mann-Whitney U test
    u_statistic, p_value = stats.mannwhitneyu(positive, negative, alternative='two-sided')
    if p_value > 1:  # Skip if the difference is not significant
        continue
    
    print(f'p-value {p_value}')
    print(f'{label_set} positive mean: {positive.mean():.4f}', 'std', f'{positive.std():.4f}', 'negative mean', f'{negative.mean():.4f}', 'std', f'{negative.std():.4f}')
    
    # Prepare data for the violin plot
    data = pd.DataFrame({
        'Status': ['Senescence'] * len(positive) + ['Non-senescence'] * len(negative),
        'Num_Pixel': np.concatenate([positive, negative])
    })

    # Create the violin plot
    plt.figure(figsize=(8, 7))
    plt.rcParams.update({'font.size': 8})
    plt.rcParams.update({'font.family': 'Arial'})

    sns.violinplot(x='Status', y='Num_Pixel', data=data, palette=["#EB382E", "#B7D6F0"], inner="box", linewidth=1.5, alpha=0.8)

    # Add p-value annotation to the plot
    plt.text(0.5, 0.95, f'p = {p_value:.2e}', horizontalalignment='center', verticalalignment='center', transform=plt.gca().transAxes)

    # Labeling the plot
    plt.ylabel('Number of pixels (Raman)')
    plt.title(label_set + ' (global)')
    
    # Hide the top and right spines
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)

    # Save the plot
    plt.savefig(f'figures/{sample}/{save_dir}/Old_SvsNS_{label_set}_global_size.pdf')
    plt.show()

# Cell Type Specific Analysis for S (O vs Y)
for label_set, tiltle_set in zip(label_sets, tiltle_sets):
   
    selected_cell_mapping = all_cell_mapping[all_cell_mapping['sample_type'] == 'O']
    cell_types = set(selected_cell_mapping['cell_type'])

    for cell_type in cell_types: 

        selected_cell_type_mapping = selected_cell_mapping[selected_cell_mapping['cell_type'] == cell_type]

        if len(selected_cell_type_mapping) < 3:
            continue

        labels = selected_cell_type_mapping[label_set].values * 1 # S vs NS
        features = selected_cell_type_mapping['num_pixel']

        positive = features[labels == 1]
        negative = features[labels == 0]
        
        if len(positive) < 3 or len(negative) < 3:
            continue

        # Perform Mann-Whitney U test
        u_statistic, p_value = stats.mannwhitneyu(positive, negative, alternative='two-sided')
        if p_value > 1:  # Skip if the difference is not significant
            continue
        
        print(f'p-value {p_value}')
        print(f'{label_set} positive mean: {positive.mean():.4f}', 'std', f'{positive.std():.4f}', 'negative mean', f'{negative.mean():.4f}', 'std', f'{negative.std():.4f}')
        
        # Prepare data for the violin plot
        data = pd.DataFrame({
            'Status': ['Senescence'] * len(positive) + ['Non-senescence'] * len(negative),
            'Num_Pixel': np.concatenate([positive, negative])
        })

        # Create the violin plot
        plt.figure(figsize=(8, 7))
        plt.rcParams.update({'font.size': 8})
        plt.rcParams.update({'font.family': 'Arial'})

        sns.violinplot(x='Status', y='Num_Pixel', data=data, palette=["#EB382E", "#B7D6F0"], inner="box", linewidth=1.5, alpha=0.8)

        # Add p-value annotation to the plot
        plt.text(0.5, 0.95, f'p = {p_value:.2e}', horizontalalignment='center', verticalalignment='center', transform=plt.gca().transAxes)

        # Labeling the plot
        plt.ylabel('Number of pixels (Raman)')
        plt.title(label_set + ' (' + cell_type + ')')
        
        # Hide the top and right spines
        plt.gca().spines['top'].set_visible(False)
        plt.gca().spines['right'].set_visible(False)

        # Save the plot
        if '/' in cell_type:
            cell_type = cell_type.replace('/', '_')

        plt.savefig(f'figures/{sample}/{save_dir}/Old_SvsNS_{label_set}_{cell_type}_size.pdf')
        plt.show()


In [ ]:
# All sample, O vs Y, intensity


from scipy import stats
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

label_sets = [ 'all']
tiltle_sets = [ 'all']

for label_set, tiltle_set in zip(label_sets, tiltle_sets):
   
    selected_cell_mapping = all_cell_mapping
    selected_cell_raman = all_cell_raman

    if len(selected_cell_mapping) < 3:
        continue

    labels = selected_cell_mapping['sample_type'] == 'O'
    features = selected_cell_raman.mean(axis=1).astype(float)

    positive = features[labels == 1]
    negative = features[labels == 0]
    
    if len(positive) < 3 or len(negative) < 3:
        continue

    # Perform Mann-Whitney U test
    u_statistic, p_value = stats.mannwhitneyu(positive, negative, alternative='two-sided')
    if p_value > 1:  # Skip if the difference is not significant
        continue
    
    print(f'p-value {p_value}')
    print(f'{label_set} positive mean: {positive.mean():.4f}', 'std', f'{positive.std():.4f}', 'negative mean', f'{negative.mean():.4f}', 'std', f'{negative.std():.4f}')
    
    # Prepare data for the violin plot
    data = pd.DataFrame({
        'Status': ['Old'] * len(positive) + ['Young'] * len(negative),
        'Intensity': np.concatenate([positive, negative])
    })

    # Create the violin plot
    plt.figure(figsize=(8, 7))
    plt.rcParams.update({'font.size': 8})
    plt.rcParams.update({'font.family': 'Arial'})

    sns.violinplot(x='Status', y='Intensity', data=data, palette=["#EB382E", "#B7D6F0"], inner="box", linewidth=1.5, alpha=0.8)

    # Add p-value annotation to the plot
    plt.text(0.5, 0.95, f'p = {p_value:.2e}', horizontalalignment='center', verticalalignment='center', transform=plt.gca().transAxes)

    # Labeling the plot
    plt.ylabel('Raman pixel intensity')
    plt.xlabel('Status')
    plt.title(label_set + ' (global)')
    
    # Hide the top and right spines
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)

    # Set x-axis limits
    # plt.ylim([20, 200])

    # Save the plot
    if not os.path.exists(f'figures/{sample}/{save_dir}'):
        os.makedirs(f'figures/{sample}/{save_dir}')
    plt.savefig(f'figures/{sample}/{save_dir}/All_OvsY_{label_set}_global_intensity.pdf')
    plt.show()

# Cell Type Specific Analysis for Old (S vs NS)
for label_set, tiltle_set in zip(label_sets, tiltle_sets):
   
    selected_cell_mapping = all_cell_mapping
    selected_cell_raman = all_cell_raman
    cell_types = set(selected_cell_mapping['cell_type'])

    for cell_type in cell_types: 

        selected_cell_type_mapping = selected_cell_mapping[selected_cell_mapping['cell_type'] == cell_type]
        selected_cell_type_raman = selected_cell_raman.loc[selected_cell_type_mapping.index]

        if len(selected_cell_type_mapping) < 3:
            continue

        labels = selected_cell_type_mapping['sample_type'] == 'O'
        features = selected_cell_type_raman.mean(axis=1).astype(float)

        positive = features[labels == 1]
        negative = features[labels == 0]
        
        if len(positive) < 3 or len(negative) < 3:
            continue

        # Perform Mann-Whitney U test
        u_statistic, p_value = stats.mannwhitneyu(positive, negative, alternative='two-sided')
        if p_value > 1:  # Skip if the difference is not significant
            continue
        
        print(f'p-value {p_value}')
        print(f'{label_set} positive mean: {positive.mean():.4f}', 'std', f'{positive.std():.4f}', 'negative mean', f'{negative.mean():.4f}', 'std', f'{negative.std():.4f}')
        
        # Prepare data for the violin plot
        data = pd.DataFrame({
            'Status': ['Old'] * len(positive) + ['Young'] * len(negative),
            'Intensity': np.concatenate([positive, negative])
        })

        # Create the violin plot
        plt.figure(figsize=(8, 7))
        plt.rcParams.update({'font.size': 8})
        plt.rcParams.update({'font.family': 'Arial'})

        sns.violinplot(x='Status', y='Intensity', data=data, palette=["#EB382E", "#B7D6F0"], inner="box", linewidth=1.5, alpha=0.8)

        # Add p-value annotation to the plot
        plt.text(0.5, 0.95, f'p = {p_value:.2e}', horizontalalignment='center', verticalalignment='center', transform=plt.gca().transAxes)

        # Labeling the plot
        plt.ylabel('Raman pixel intensity')
        plt.xlabel('Status')
        plt.title(label_set + ' (' + cell_type + ')')

        # Hide the top and right spines
        plt.gca().spines['top'].set_visible(False)
        plt.gca().spines['right'].set_visible(False)

        # Set x-axis limits
        # plt.ylim([20, 200])

        # Save the plot
        if '/' in cell_type:
            cell_type = cell_type.replace('/', '_')

        if not os.path.exists(f'figures/{sample}/{save_dir}'):
            os.makedirs(f'figures/{sample}/{save_dir}')

        plt.savefig(f'figures/{sample}/{save_dir}/All_OvsY_{cell_type}_intensity.pdf')
        plt.show()
